# Obesity Classification — Lifestyle-Only Analysis
## EDA + Feature Engineering (Weight & Height Removed Before Everything)

**Research question:** Can we predict obesity level from *behavioral and lifestyle* data alone?

We drop `Weight` and `Height` on the very first line — before any EDA, any chart, any cleaning.
This eliminates the trivial data leak and forces every chart and every model to work from
diet, physical activity, transport, and habits — the features that actually matter clinically.

---

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
SEED = 42
np.random.seed(SEED)

In [ ]:
DATA_PATH   = "../models/ObesityDataSet_raw_and_data_sinthetic.csv"
TARGET      = "NObeyesdad"
TARGET_ORDER = {
    "Insufficient_Weight": 0, "Normal_Weight":       1,
    "Overweight_Level_I":  2, "Overweight_Level_II": 3,
    "Obesity_Type_I":      4, "Obesity_Type_II":     5,
    "Obesity_Type_III":    6,
}
TARGET_LABELS = list(TARGET_ORDER.keys())
PALETTE       = sns.color_palette("viridis", n_colors=7)

FREQ_ORDER  = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
SMOTE_CLIPS = {"FCVC": (1, 3), "NCP": (1, 4), "CH2O": (1, 3), "FAF": (0, 3), "TUE": (0, 2)}
SMOTE_ORDS  = list(SMOTE_CLIPS.keys())
BIN_COLS    = ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]
ORD_CAT     = ["CAEC", "CALC"]
NOM_CAT     = ["Gender", "MTRANS"]

---
## 1. The Data Leakage Problem

The target `NObeyesdad` is defined **directly by WHO BMI thresholds**:

| Class | WHO definition |
|-------|---------------|
| Insufficient_Weight | BMI < 18.5 |
| Normal_Weight | 18.5 ≤ BMI < 25 |
| Overweight_Level_I / II | 25 ≤ BMI < 35 |
| Obesity_Type_I / II / III | BMI ≥ 35 |

And **BMI = Weight / Height²**.

Including `Weight` or `Height` means any model will learn to recompute BMI and look up the label —
achieving ~97% accuracy without learning anything about lifestyle.
This is **definitional data leakage**: the features encode the answer by construction.

A model trained on Weight + Height cannot tell us *why* someone becomes obese or *what behaviours* predict it.
That is the scientifically interesting and clinically actionable question.

**We drop both columns before the first chart.**

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Original shape: {df_raw.shape}")

# ─── Drop Weight and Height BEFORE any EDA ─────────────────────────────────
df = df_raw.drop(columns=["Weight", "Height"])

assert "Weight" not in df.columns and "Height" not in df.columns
print(f"After drop: {df.shape}")
print(f"\nFeatures ({len(df.columns)-1}):")
for c in df.columns:
    if c != TARGET:
        print(f"  {c}  [{df[c].dtype}]")
df.head()

---
## 2. Exploratory Data Analysis

### 2.1 Target Distribution

The dataset contains 7 ordered obesity classes balanced by SMOTE augmentation (77% synthetic rows).
The good news: no class imbalance problem. The challenge: distinguishing 7 classes from behavioral
data alone — without the anthropometric shortcut.

In [ ]:
counts = df[TARGET].value_counts().reindex(TARGET_LABELS)
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(TARGET_LABELS, counts.values, color=PALETTE, edgecolor="white", linewidth=0.8)
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_ylabel("Count")
ax.set_title("Target Distribution — 7 Ordered Obesity Classes  (n = 2,111)", fontsize=13, fontweight="bold")
ax.set_xticklabels(TARGET_LABELS, rotation=25, ha="right")
plt.tight_layout()
plt.show()

### 2.2 SMOTE-Interpolated Ordinals

Five columns were originally **integer survey responses** (e.g., "How many meals per day: 1 / 2 / 3 / 4").
SMOTE augmented the dataset by linearly interpolating between respondents, turning integers like `2`
into floats like `2.784`. The red lines show the valid integer survey ticks.

**Why this matters:** treating 2.78 as meaningfully different from 2.80 adds noise to ordinal data.
We will recover the integers in the cleaning step.

In [ ]:
smote_ticks = {"FCVC": [1,2,3], "NCP": [1,2,3,4], "CH2O": [1,2,3], "FAF": [0,1,2,3], "TUE": [0,1,2]}
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, col in zip(axes, SMOTE_ORDS):
    ax.hist(df[col], bins=50, color="steelblue", edgecolor="white", alpha=0.85)
    for t in smote_ticks[col]:
        ax.axvline(t, color="crimson", linewidth=1.3, linestyle="--")
    ax.set_title(col, fontsize=11)
    ax.set_xlabel("value  (red = integer ticks)")
plt.suptitle("SMOTE Artifact: Values Spread Between Integer Survey Ticks", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### 2.3 Physical Activity (FAF) — The Strongest Modifiable Factor

`FAF` measures how many days per week the respondent is physically active (0 = none, 3 = 4–5 days).
It is the single behavioral feature with the clearest monotone separation across obesity classes.

**Observation:** Median FAF drops as obesity increases. Obese Type III subjects exercise significantly less
than Insufficient Weight subjects — this is the most actionable signal in the dataset.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box: FAF by class
sns.boxplot(data=df, x=TARGET, y="FAF", order=TARGET_LABELS,
            palette=PALETTE, ax=axes[0], linewidth=0.8)
axes[0].set_xticklabels(TARGET_LABELS, rotation=25, ha="right")
axes[0].set_title("Physical Activity Frequency by Obesity Level", fontsize=11, fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("FAF (days/week, raw continuous)")

# Strip: TUE by class
sns.boxplot(data=df, x=TARGET, y="TUE", order=TARGET_LABELS,
            palette=PALETTE, ax=axes[1], linewidth=0.8)
axes[1].set_xticklabels(TARGET_LABELS, rotation=25, ha="right")
axes[1].set_title("Technology Use (Screen Time) by Obesity Level", fontsize=11, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("TUE (hours/day, raw continuous)")

plt.tight_layout()
plt.show()

# Spearman correlation
y_num = df[TARGET].map(TARGET_ORDER)
rho_faf, p_faf = stats.spearmanr(df["FAF"], y_num)
rho_tue, p_tue = stats.spearmanr(df["TUE"], y_num)
print(f"Spearman rho  FAF vs obesity: {rho_faf:+.3f}  (p={p_faf:.2e})")
print(f"Spearman rho  TUE vs obesity: {rho_tue:+.3f}  (p={p_tue:.2e})")

### 2.4 Eating Behavior: CAEC × FAVC

`CAEC` is how frequently the respondent eats *between* main meals (no / Sometimes / Frequently / Always).
`FAVC` flags frequent consumption of high-calorie foods.

Together they capture **snacking intensity and food quality** — the two most directly controllable
dietary risk factors. The cross-plot reveals whether habitual snackers *also* tend to eat high-calorie food.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CAEC stacked bar
ct_caec = (df.groupby(["CAEC", TARGET]).size()
             .unstack(TARGET).reindex(columns=TARGET_LABELS, fill_value=0))
ct_caec_norm = ct_caec.div(ct_caec.sum(axis=1), axis=0)
caec_order = ["no", "Sometimes", "Frequently", "Always"]
ct_caec_norm.reindex(caec_order).plot(
    kind="bar", stacked=True, ax=axes[0], colormap="viridis",
    edgecolor="white", linewidth=0.3, legend=False)
axes[0].set_title("Snacking Frequency (CAEC) — Share per Obesity Class", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Eating Between Meals")
axes[0].tick_params(axis="x", rotation=20)

# FAVC × CAEC cross-tab heat
cross = pd.crosstab(df["CAEC"].map({"no":"no","Sometimes":"Sometimes","Frequently":"Frequently","Always":"Always"}),
                    df["FAVC"])
cross_pct = cross.div(cross.sum(axis=1), axis=0)
sns.heatmap(cross_pct.reindex(caec_order), annot=True, fmt=".2f",
            cmap="YlOrRd", linewidths=0.4, ax=axes[1])
axes[1].set_title("CAEC × FAVC — Snacking Rate by High-Calorie Food Habit", fontsize=11, fontweight="bold")
axes[1].set_xlabel("FAVC (Frequent High-Calorie Food)")
axes[1].set_ylabel("CAEC (Eating Between Meals)")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=7,
           title="Obesity Level", bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.show()

### 2.5 Transport Behaviour — A Proxy for Habitual Activity

`MTRANS` captures how the respondent usually gets around: Walking, Bike, Motorbike, Automobile,
or Public_Transportation. Walking and Bike represent **active transport** — built-in physical
activity that requires no gym. Automobile and Motorbike represent **passive transport** — sedentary
default behaviours.

This feature is particularly interesting because it reflects *structural lifestyle choice* rather
than a deliberate fitness choice. Active commuters exercise without thinking about it.

In [ ]:
# Active vs Passive
df["transport_active_eda"] = df["MTRANS"].isin(["Walking", "Bike"]).astype(int)
group_map = {0: "Passive (Car/Moto/Public)", 1: "Active (Walk/Bike)"}

ct = (df.groupby(["transport_active_eda", TARGET]).size()
        .unstack(TARGET).reindex(columns=TARGET_LABELS, fill_value=0))
ct_norm = ct.div(ct.sum(axis=1), axis=0)
ct_norm.index = [group_map[i] for i in ct_norm.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ct_norm.plot(kind="bar", stacked=True, ax=axes[0], colormap="viridis",
             edgecolor="white", linewidth=0.3, legend=False)
axes[0].set_title("Active vs Passive Transport — Obesity Share", fontsize=11, fontweight="bold")
axes[0].tick_params(axis="x", rotation=15)
axes[0].set_xlabel("Transport Group")

# By MTRANS category
ct2 = (df.groupby(["MTRANS", TARGET]).size()
         .unstack(TARGET).reindex(columns=TARGET_LABELS, fill_value=0))
ct2_norm = ct2.div(ct2.sum(axis=1), axis=0)
obesity_cols = [c for c in TARGET_LABELS if "Obesity" in c]
ct2_norm["total_obese"] = ct2_norm[obesity_cols].sum(axis=1)
ct2_norm = ct2_norm.sort_values("total_obese")
ct2_norm.drop(columns="total_obese").plot(
    kind="bar", stacked=True, ax=axes[1], colormap="viridis",
    edgecolor="white", linewidth=0.3, legend=False)
axes[1].set_title("All Transport Modes — Sorted by Total Obesity %", fontsize=11, fontweight="bold")
axes[1].tick_params(axis="x", rotation=25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=7,
           title="Obesity Level", bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.show()

df.drop(columns="transport_active_eda", inplace=True)

### 2.6 Lifestyle Region: Activity × Diet

Combining FAF and FAVC creates three lifestyle profiles:

| Region | Condition | Interpretation |
|--------|-----------|----------------|
| **Healthy** | FAF ≥ 2 AND no high-cal food | Active + clean diet |
| **High Risk** | FAF < 1 AND eats high-cal food | Inactive + poor diet |
| **Moderate** | Everything else | Mixed habits |

This cross-feature reveals the *joint* effect — being active but eating badly vs being inactive but eating well.

In [ ]:
def lifestyle_region(row):
    if row["FAF"] >= 2 and row["FAVC"] == "no":
        return "Healthy"
    elif row["FAF"] < 1 and row["FAVC"] == "yes":
        return "High Risk"
    return "Moderate"

df["lifestyle_region_eda"] = df.apply(lifestyle_region, axis=1)

ct = (df.groupby(["lifestyle_region_eda", TARGET]).size()
        .unstack(TARGET).reindex(columns=TARGET_LABELS, fill_value=0))
ct_norm = ct.div(ct.sum(axis=1), axis=0)
region_order = ["Healthy", "Moderate", "High Risk"]
ct_norm = ct_norm.reindex(region_order)

fig, ax = plt.subplots(figsize=(11, 5))
ct_norm.plot(kind="bar", stacked=True, ax=ax, colormap="viridis",
             edgecolor="white", linewidth=0.3)
ax.set_title("Obesity Distribution by Lifestyle Region", fontsize=12, fontweight="bold")
ax.set_xlabel("Lifestyle Region")
ax.tick_params(axis="x", rotation=10)
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left", title="Obesity Level")
plt.tight_layout()
plt.show()

obesity_share = ct_norm[[c for c in TARGET_LABELS if "Obesity" in c]].sum(axis=1)
print("Total obesity share by region:")
for r, v in obesity_share.items():
    print(f"  {r:<12s}: {v*100:.1f}%")

df.drop(columns="lifestyle_region_eda", inplace=True)

### 2.7 Screen Time × Physical Activity — The Digital Age Paradox

High screen time (TUE > 3h/day) combined with low physical activity (FAF < 1 day/week) defines
the **sedentary digital lifestyle** — the most concerning modern risk pattern.

This interaction captures something neither TUE nor FAF captures alone: the person who is both
inactive AND highly sedentary outside of work/school.

In [ ]:
def screen_activity(row):
    if row["TUE"] > 3 and row["FAF"] < 1:
        return "High Screen / Inactive"
    elif row["TUE"] < 2 and row["FAF"] >= 2:
        return "Low Screen / Active"
    return "Balanced"

df["screen_act_eda"] = df.apply(screen_activity, axis=1)

ct = (df.groupby(["screen_act_eda", TARGET]).size()
        .unstack(TARGET).reindex(columns=TARGET_LABELS, fill_value=0))
ct_norm = ct.div(ct.sum(axis=1), axis=0)
sa_order = ["Low Screen / Active", "Balanced", "High Screen / Inactive"]
ct_norm = ct_norm.reindex(sa_order)

fig, ax = plt.subplots(figsize=(11, 5))
ct_norm.plot(kind="bar", stacked=True, ax=ax, colormap="viridis",
             edgecolor="white", linewidth=0.3)
ax.set_title("Screen Time × Activity Interaction — Obesity Distribution", fontsize=12, fontweight="bold")
ax.tick_params(axis="x", rotation=10)
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left", title="Obesity Level")
plt.tight_layout()
plt.show()

obesity_share = ct_norm[[c for c in TARGET_LABELS if "Obesity" in c]].sum(axis=1)
print("Total obesity share by screen-activity group:")
for r, v in obesity_share.items():
    print(f"  {r:<30s}: {v*100:.1f}%")

df.drop(columns="screen_act_eda", inplace=True)

### 2.8 Can You Outrun Your Genes?

`family_history_with_overweight` is a genetic predisposition flag. The key question:
does physical activity offset genetic risk?

We cross family history with activity level to create four risk quadrants. This shows
whether lifestyle intervention can compensate for genetic risk — a clinically critical question.

In [ ]:
def risk_group(row):
    fam  = row["family_history_with_overweight"] == "yes"
    act  = row["FAF"] >= 2
    if fam  and not act: return "Genetic + Inactive"
    if fam  and act:     return "Genetic + Active"
    if not fam and not act: return "No Genetic + Inactive"
    return "No Genetic + Active"

df["risk_grp_eda"] = df.apply(risk_group, axis=1)

ct = (df.groupby(["risk_grp_eda", TARGET]).size()
        .unstack(TARGET).reindex(columns=TARGET_LABELS, fill_value=0))
ct_norm = ct.div(ct.sum(axis=1), axis=0)
rg_order = ["Genetic + Inactive", "Genetic + Active",
            "No Genetic + Inactive", "No Genetic + Active"]
ct_norm = ct_norm.reindex([r for r in rg_order if r in ct_norm.index])

fig, ax = plt.subplots(figsize=(12, 5))
ct_norm.plot(kind="bar", stacked=True, ax=ax, colormap="viridis",
             edgecolor="white", linewidth=0.3)
ax.set_title("Genetic Risk × Physical Activity — Four Quadrants", fontsize=12, fontweight="bold")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left", title="Obesity Level")
plt.tight_layout()
plt.show()

obesity_share = ct_norm[[c for c in TARGET_LABELS if "Obesity" in c]].sum(axis=1)
print("Total obesity share by risk quadrant:")
for r, v in obesity_share.items():
    print(f"  {r:<30s}: {v*100:.1f}%")
print()
print("Key insight: genetic predisposition raises baseline risk, but activity reduces it even for carriers.")

df.drop(columns="risk_grp_eda", inplace=True)

### 2.9 Generational Risk Behaviors

Risk behaviour patterns differ by age group. We compare Gen Z (≤ 25) with older adults (> 35)
across five modifiable risk behaviors: alcohol, low water intake, snacking, inactivity, and high screen time.

This shows **which generation's behaviors** most predict obesity — relevant for targeting interventions.

In [ ]:
def check_risks(sub_df, label):
    sub = sub_df.copy()
    sub["Obese"] = sub[TARGET].str.startswith("Obesity")
    results = {}
    for col, flag, name in [
        ("CALC",  lambda s: s.isin(["Frequently","Always"]),     "High Alcohol"),
        ("CH2O",  lambda s: s < 2,                               "Low Water"),
        ("CAEC",  lambda s: s.isin(["Frequently","Always"]),     "Frequent Snacking"),
        ("FAF",   lambda s: s < 1,                               "Inactive"),
        ("TUE",   lambda s: s > 3,                               "High Screen"),
    ]:
        sub["_flag"] = flag(sub[col])
        g = sub.groupby("_flag")["Obese"].mean()
        results[name] = g.get(True, 0) * 100
    return pd.Series(results, name=label)

genz  = df[df["Age"] <= 25]
older = df[df["Age"] >  35]

risk_df = pd.DataFrame([
    check_risks(genz,  "Gen Z (≤25)"),
    check_risks(older, "Older (>35)"),
])

fig, ax = plt.subplots(figsize=(10, 4))
risk_df.T.plot(kind="bar", ax=ax, color=["#E07A5F", "#3D405B"], edgecolor="white", linewidth=0.6)
ax.set_title("% Obese Among People WITH Each Risk Behavior — Gen Z vs Older", fontsize=12, fontweight="bold")
ax.set_ylabel("% Obese")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

display(risk_df.round(1))

### 2.10 Spearman Correlation — Statistical Signal of Each Feature

Spearman rank correlation measures monotone association without assuming linearity.
We use it here because our target is ordinal (not interval) and many features are ordinal.

This chart ranks every lifestyle feature by its statistical relationship to obesity level.
It answers: **which single features carry the most predictive signal?**

In [ ]:
y_num = df[TARGET].map(TARGET_ORDER)

# encode categoricals for correlation
enc = df.copy()
for col in BIN_COLS:
    enc[col] = (enc[col] == "yes").astype(int)
for col in ORD_CAT:
    enc[col] = enc[col].map(FREQ_ORDER)
enc["Gender"]  = (enc["Gender"] == "Male").astype(int)
enc["MTRANS"]  = enc["MTRANS"].map({"Walking":0,"Bike":1,"Public_Transportation":2,
                                     "Motorbike":3,"Automobile":4})

feature_cols = [c for c in enc.columns if c != TARGET]
rho_vals = {}
for col in feature_cols:
    rho, p = stats.spearmanr(enc[col], y_num)
    rho_vals[col] = (rho, p)

rho_series = pd.Series({k: v[0] for k, v in rho_vals.items()}).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#d73027" if v > 0 else "#4575b4" for v in rho_series.values]
ax.barh(rho_series.index, rho_series.values, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Spearman rho with ordinal obesity level")
ax.set_title("Lifestyle Feature Correlations with Obesity Class\n(red = positive risk, blue = protective)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Top 5 positive (obesity risk factors):")
for col, val in rho_series.sort_values(ascending=False).head(5).items():
    p = rho_vals[col][1]
    print(f"  {col:<42s} rho={val:+.3f}  p={p:.2e}")

print("\nTop 5 negative (protective factors):")
for col, val in rho_series.sort_values().head(5).items():
    p = rho_vals[col][1]
    print(f"  {col:<42s} rho={val:+.3f}  p={p:.2e}")

### 2.11 EDA Summary — Key Findings

| Feature | Direction | Insight |
|---------|-----------|---------|
| `family_history` | Positive | Strongest single predictor — genetic load |
| `CAEC` (snacking) | Positive | More snacking → higher obesity — modifiable |
| `FAVC` (high-cal food) | Positive | Diet quality matters |
| `FAF` (physical activity) | **Negative** | The most important protective factor |
| `MTRANS` | Mixed | Active transport (Walk/Bike) is protective |
| `CH2O` (water) | Negative | Hydration correlates with health-conscious behaviour |
| `SMOKE` | Near zero | Smoking does not strongly predict obesity class in this sample |
| `SCC` (calorie tracking) | Mixed | Counter-intuitive — calorie trackers span all obesity levels |

**Without Weight and Height, the task is genuinely hard (~65-80% F1 expected).**
The signal exists, but it is noisy and multi-dimensional — which is what makes it scientifically interesting.

---

---
## 3. Data Cleaning

### 3.1 SMOTE Ordinal Recovery

We clip each SMOTE column to its valid integer range and round. This recovers the original
survey categories without losing any real respondent data (original 498 rows already had integers).

In [ ]:
for col, (lo, hi) in SMOTE_CLIPS.items():
    df[f"{col}_int"] = np.clip(df[col], lo, hi).round().astype(int)

print("Recovered integer columns:")
for col in SMOTE_ORDS:
    raw_unique = sorted(df[col].unique())[:6]
    int_unique = sorted(df[f"{col}_int"].unique())
    print(f"  {col:<6s}  raw sample: {raw_unique} ...  →  integer: {int_unique}")

### 3.2 Outlier Check — Age (only continuous feature left)

After removing Weight and Height, `Age` is the only true continuous measurement remaining.
We verify it has no anomalous values.

In [ ]:
q1, q3 = df["Age"].quantile(0.25), df["Age"].quantile(0.75)
iqr = q3 - q1
outliers = df[(df["Age"] < q1 - 3*iqr) | (df["Age"] > q3 + 3*iqr)]
print(f"Age: min={df['Age'].min():.0f}  max={df['Age'].max():.0f}  outliers (3×IQR): {len(outliers)}")
print("No outlier removal needed.")

---
## 4. Feature Engineering

We build three layers of features, each with a clear medical or behavioural motivation.
The rule: **if we cannot explain the feature in plain language, we do not build it.**

| Layer | What we build | Why |
|-------|--------------|-----|
| **Encoding** | Binary (0/1), ordinal integers, one-hot | Machine-readable form of existing features |
| **Domain composites** | Caloric risk, health score, lifestyle region | Combine related features the model might miss |
| **Interactions** | FAF × TUE, risk quadrant | Non-linear joint effects between key pairs |

### 4.1 Caloric Risk Score

Inspired by the nutritional epidemiology concept of *energy imbalance*, this score combines:
- **Meal load** (NCP_int): more meals = higher caloric intake
- **Snacking frequency** (CAEC ordinal): between-meal eating adds untracked calories
- **High-calorie food** (FAVC binary, weighted ×2): strong quality signal

`caloric_risk = NCP_int + CAEC_ord + 2 × FAVC_bin`

Higher scores indicate higher estimated net caloric intake relative to activity.

In [ ]:
# Encode first
for col in BIN_COLS:
    df[f"{col}_bin"] = (df[col] == "yes").astype(int)
for col in ORD_CAT:
    df[f"{col}_ord"] = df[col].map(FREQ_ORDER).astype(int)

df["caloric_risk"] = df["NCP_int"] + df["CAEC_ord"] + 2 * df["FAVC_bin"]

# Verify it correlates with target
y_num = df[TARGET].map(TARGET_ORDER)
rho_cr, p_cr = stats.spearmanr(df["caloric_risk"], y_num)
print(f"Caloric Risk Score — Spearman rho: {rho_cr:+.3f}  (p={p_cr:.2e})")
print(f"Compare: raw NCP rho = {stats.spearmanr(df['NCP_int'], y_num)[0]:+.3f}")
print(f"         raw CAEC rho = {stats.spearmanr(df['CAEC_ord'], y_num)[0]:+.3f}")
print(f"         raw FAVC rho = {stats.spearmanr(df['FAVC_bin'], y_num)[0]:+.3f}")
print()
print("Composite > any individual component — the combination has more signal.")

### 4.2 Behavioral Composite Features

Five composite features encode structural lifestyle patterns that no single column captures:

In [ ]:
def add_composites(df):
    out = df.copy()

    # Age group (0 = teen ... 4 = senior)
    out["age_group"] = pd.cut(out["Age"],
        bins=[0, 18, 25, 35, 50, float("inf")],
        labels=range(5), include_lowest=True).astype(int)

    # Health score: normalised sum of three protective behaviors
    out["health_score"] = (out["FAF_int"]/3 + out["CH2O_int"]/3 + out["FCVC_int"]/3)

    # Transport: active commute (0 = passive, 1 = active)
    out["transport_active"] = out["MTRANS"].isin(["Walking", "Bike"]).astype(int)

    # Screen–activity: 0 = active+low screen, 1 = balanced, 2 = inactive+high screen
    def sa(row):
        if row["TUE_int"] > 1 and row["FAF_int"] < 1: return 2
        if row["TUE_int"] < 2 and row["FAF_int"] >= 2: return 0
        return 1
    out["screen_activity"] = out.apply(sa, axis=1)

    # Risk quadrant: family history × activity (0 = best ... 3 = worst)
    def rq(row):
        fam = row["family_history_with_overweight_bin"]
        act = row["FAF_int"] >= 2
        if   fam == 0 and act:     return 0
        elif fam == 1 and act:     return 1
        elif fam == 0 and not act: return 2
        else:                      return 3
    out["risk_quadrant"] = out.apply(rq, axis=1)

    # Diet type: 0 = IF-like (few meals + no snacking), 1 = controlled, 2 = uncontrolled
    def dt(row):
        if row["NCP_int"] <= 2 and row["CAEC_ord"] == 0: return 0
        if row["FAVC_bin"] == 0 and row["CAEC_ord"] <= 1: return 1
        return 2
    out["diet_type"] = out.apply(dt, axis=1)

    # Lifestyle region: 0 = healthy, 1 = moderate, 2 = high risk
    def lr(row):
        if row["FAF_int"] >= 2 and row["FAVC_bin"] == 0: return 0
        if row["FAF_int"] < 1  and row["FAVC_bin"] == 1: return 2
        return 1
    out["lifestyle_region"] = out.apply(lr, axis=1)

    # Key interaction: FAF × TUE (inverse relationship: active + low screen = low risk)
    out["FAF_x_TUE"] = out["FAF_int"] * out["TUE_int"]

    return out

df = add_composites(df)

new_feats = ["age_group", "health_score", "transport_active", "screen_activity",
             "risk_quadrant", "diet_type", "lifestyle_region", "FAF_x_TUE"]
print("New composite features:")
for f in new_feats:
    rho, _ = stats.spearmanr(df[f], y_num)
    print(f"  {f:<22s}  rho={rho:+.3f}  range=[{df[f].min():.2f}, {df[f].max():.2f}]")

In [ ]:
# One-hot for nominal categoricals
df = pd.get_dummies(df, columns=NOM_CAT, drop_first=False, dtype=int)

ohe_cols = [c for c in df.columns if any(c.startswith(p) for p in ["Gender_", "MTRANS_"])]
print(f"One-hot columns created: {ohe_cols}")

### 4.3 Final Feature Set

All raw strings → numeric. All SMOTE floats → integers. All domain knowledge → explicit features.

In [ ]:
DROP_COLS = (
    SMOTE_ORDS             # replaced by _int versions
    + BIN_COLS             # replaced by _bin versions
    + ORD_CAT              # replaced by _ord versions
    + [TARGET]
)
raw_nominal = [c for c in ["Gender", "MTRANS"] if c in df.columns]  # already one-hotted

FEATURES = [
    "Age", "age_group",
    "FCVC_int", "NCP_int", "CH2O_int", "FAF_int", "TUE_int",
    "family_history_with_overweight_bin", "FAVC_bin", "SMOKE_bin", "SCC_bin",
    "CAEC_ord", "CALC_ord",
    "caloric_risk", "health_score", "transport_active",
    "screen_activity", "risk_quadrant", "diet_type", "lifestyle_region",
    "FAF_x_TUE",
] + [c for c in df.columns if c.startswith("Gender_") or c.startswith("MTRANS_")]

X = df[FEATURES]
y = df[TARGET].map(TARGET_ORDER)

print(f"Final feature matrix: {X.shape}")
print(f"All numeric: {X.dtypes.nunique() <= 2}")
print(f"Zero nulls:  {X.isnull().sum().sum() == 0}")
print()

dtype_info = pd.DataFrame({
    "dtype": X.dtypes,
    "min":   X.min().round(3),
    "max":   X.max().round(3),
    "mean":  X.mean().round(3),
})
display(dtype_info)

# Final Spearman correlations
final_rho = X.corrwith(y).abs().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 8))
final_rho.plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.invert_yaxis()
ax.set_xlabel("|Spearman rho| with ordinal target")
ax.set_title("Final Feature Correlations — Lifestyle Only (No Weight/Height)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Top 10 features by |Spearman rho|:")
for rank, (feat, val) in enumerate(final_rho.head(10).items(), 1):
    rho_signed = X[feat].corr(y, method="spearman")
    print(f"  {rank:2d}. {feat:<40s} rho={rho_signed:+.3f}")

---
## 5. What We Built

**Removed:** `Weight`, `Height` (and all BMI derivatives) — data leakage eliminated.

**14 raw features** → **28 engineered features** across three layers:

| Layer | Count | Examples |
|-------|-------|---------|
| Encoded originals | 17 | `FCVC_int`, `CAEC_ord`, `Gender_Male`, `MTRANS_Walking` |
| Domain composites | 7 | `caloric_risk`, `health_score`, `risk_quadrant`, `lifestyle_region` |
| Interactions | 1 | `FAF_x_TUE` |

**Hand-off to modeling:** `X` (28 features) + `y` (0–6 ordinal target) are ready.
The modeling notebook will compare what happens with different subsets of these features
and which model × feature combination produces the best honest performance.

→ See `obesity_modeling_comparison.ipynb`